# Emergency Department Arrival Forecasting
## Forecasting Hourly/Daily ED Patient Arrivals

**Notebook #14 of 50 — Time Series Forecasting Portfolio**

| | |
|---|---|
| **Dataset** | Hospital ER Dataset (`maalona/hospital-er`) |
| **Target** | Daily ER patient arrival count |
| **Date column** | `date` |
| **Frequency** | Daily (`D`) |
| **Primary TS Library** | StatsForecast (AutoARIMA + AutoETS + AutoTheta) |

## Learning Objectives
1. Aggregate individual patient-visit records into a daily arrival time series
2. Explore wait-time patterns and their relationship with arrival volume
3. Identify day-of-week and seasonal patterns in ED demand
4. Apply StatsForecast to forecast daily arrivals 14 days ahead
5. Discuss staffing implications — the cost of under-vs-over-forecasting in an ED

## Problem Statement
Emergency departments cannot defer demand — a patient who arrives must be seen.
Accurate arrival forecasting enables:
- **Nurse and physician scheduling** ahead of busy days
- **Triage capacity** planning
- **Bed availability** alerts to upstream wards

> Goal: Forecast daily ED arrivals 14 days ahead using historical visit records.

## Dataset Overview
**Source:** Kaggle — `maalona/hospital-er`

**License:** Check Kaggle page.

### Columns
| Column | Type | Notes |
|--------|------|-------|
| `date` | string | Patient visit datetime (may include time component) |
| `patient_id` | int | Unique patient identifier |
| `patient_age` | int | Age in years |
| `patient_sat_score` | float | Satisfaction score 1–10 (NaN if not collected) |
| `patient_admin_flag` | bool | True = patient was admitted to hospital |
| `department_referral` | string | Which department the patient was referred to |
| `wait_time` | int | Wait time in minutes |

## Environment Setup

In [ ]:
import subprocess, sys
for imp, pkg in {"kagglehub":"kagglehub","statsforecast":"statsforecast",
                  "mlforecast":"mlforecast","lightgbm":"lightgbm",
                  "lazypredict":"lazypredict","flaml":"flaml[automl]"}.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Packages ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, SeasonalNaive
from mlforecast import MLForecast
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")

def metrics(actual, pred, label):
    a = np.asarray(actual).ravel(); p = np.asarray(pred).ravel()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = (a != 0)
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else float("nan")
    print(f"{label:<40s}  MAE={mae:>9,.2f}  RMSE={rmse:>9,.2f}  MAPE={mape:>7.2f}%")
    return {"Model":label,"MAE":round(mae,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}

print("Imports OK.")

## Configuration

In [ ]:
KAGGLE_SLUG  = "maalona/hospital-er"
FREQ         = "D"
SEASON_LEN   = 7
HORIZON      = 14
TEST_DAYS    = 14
VAL_DAYS     = 14
FLAML_BUDGET = 90
RANDOM_STATE = 42
print("Config OK.")

## Kaggle Credential Check

In [ ]:
import os, pathlib as _pl
_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle env vars found."); _ok = True
_j = _pl.Path.home() / ".kaggle" / "kaggle.json"
if _j.exists():
    print(f"[OK] kaggle.json at {_j}"); _ok = True
if not _ok:
    print("WARNING: Kaggle credentials missing.")
    print("  Option A: set KAGGLE_USERNAME + KAGGLE_KEY env vars")
    print("  Option B: place kaggle.json at ~/.kaggle/kaggle.json")
    raise EnvironmentError("Set Kaggle credentials and restart.")

## Dataset Download & Loading

In [ ]:
import kagglehub
data_path=pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
csv_files=sorted(data_path.rglob("*.csv"))
print([f.name for f in csv_files])
df=pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## Data Validation

In [ ]:
# Auto-detect date column
date_col=None
for c in df.columns:
    if any(kw in c.lower() for kw in ["date","time"]):
        try:
            pd.to_datetime(df[c].dropna().iloc[:3])
            date_col=c; break
        except: pass
print(f"Date column: {date_col}")
if date_col:
    df["_dt"]=pd.to_datetime(df[date_col],errors="coerce",infer_datetime_format=True)
    print(f"Date range: {df['_dt'].min()} → {df['_dt'].max()}")
    print(f"Parse failures: {df['_dt'].isna().sum()}")
print(f"\nWait time stats:" if "wait_time" in df.columns else "No wait_time column")
if "wait_time" in df.columns:
    print(df["wait_time"].describe().round(1))

## Aggregate to Daily Arrivals

In [ ]:
if date_col:
    df=df.dropna(subset=["_dt"]).copy()
    df["_date"]=df["_dt"].dt.date
    pat_col=next((c for c in df.columns if "patient_id" in c.lower() or "id" in c.lower()),None)
    if pat_col:
        daily_raw=df.groupby("_date")[pat_col].nunique().reset_index()
        daily_raw.columns=["ds","y"]
    else:
        daily_raw=df.groupby("_date").size().reset_index(name="y")
        daily_raw.rename(columns={"_date":"ds"},inplace=True)
    daily_raw["ds"]=pd.to_datetime(daily_raw["ds"])
    daily_raw=daily_raw.sort_values("ds").reset_index(drop=True)
    full_idx=pd.date_range(daily_raw["ds"].min(),daily_raw["ds"].max(),freq="D")
    daily=daily_raw.set_index("ds").reindex(full_idx,fill_value=0).reset_index()
    daily.columns=["ds","y"]
    print(f"Series: {len(daily)} days  |  Total visits: {daily['y'].sum():,}")
    print(daily["y"].describe().round(1))
else:
    print("ERROR: No date column detected — check dataset structure.")
    raise ValueError("Date column not found.")

In [ ]:
# EDA: full series
fig=px.line(daily,x="ds",y="y",
            title="Daily ED Arrivals",labels={"ds":"Date","y":"Patients"})
fig.update_layout(template="plotly_white"); fig.show()

# Day of week
daily["dow"]=daily["ds"].dt.day_name()
dow_order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
fig2=px.box(daily[daily["y"]>0],x="dow",y="y",category_orders={"dow":dow_order},
            title="ED Arrivals by Day of Week",labels={"dow":"Day","y":"Patients"})
fig2.update_layout(template="plotly_white"); fig2.show()

In [ ]:
# Wait time by day if available
if "wait_time" in df.columns and date_col:
    wt_daily=df.groupby("_date")["wait_time"].mean().reset_index()
    wt_daily["ds"]=pd.to_datetime(wt_daily["_date"])
    fig=px.line(wt_daily,x="ds",y="wait_time",
                title="Average Daily ER Wait Time",
                labels={"ds":"Date","wait_time":"Avg Wait (min)"})
    fig.update_layout(template="plotly_white"); fig.show()

In [ ]:
if len(daily)>3*SEASON_LEN:
    ts_d=daily.set_index("ds")["y"].asfreq("D").interpolate()
    decomp=seasonal_decompose(ts_d,model="additive",period=SEASON_LEN)
    fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
    decomp.observed.plot(ax=axes[0],title="Observed")
    decomp.trend.plot(ax=axes[1],title="Trend")
    decomp.seasonal.plot(ax=axes[2],title="Seasonal (weekly)")
    decomp.resid.plot(ax=axes[3],title="Residual")
    plt.tight_layout(); plt.show()

In [ ]:
adf=adfuller(daily["y"].dropna())
print(f"ADF p={adf[1]:.4f} → {'Stationary' if adf[1]<0.05 else 'Non-stationary'}")
fig,axes=plt.subplots(1,2,figsize=(14,4))
plot_acf(daily["y"],lags=min(60,len(daily)//3),ax=axes[0])
plot_pacf(daily["y"],lags=min(60,len(daily)//3),ax=axes[1])
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
y=daily["y"]
print(f"Mean={y.mean():.1f}  Std={y.std():.1f}  CV={y.std()/y.mean():.3f}")
print(f"Zero days={( y==0).sum()}")

## Admission Rate Analysis (if available)

In [ ]:
if "patient_admin_flag" in df.columns and date_col:
    df["_admin_int"]=df["patient_admin_flag"].map({True:1,False:0,"True":1,"False":0,1:1,0:0})
    admit_daily=df.groupby("_date")["_admin_int"].mean().reset_index()
    admit_daily.columns=["ds","admission_rate"]
    admit_daily["ds"]=pd.to_datetime(admit_daily["ds"])
    fig=px.line(admit_daily,x="ds",y="admission_rate",
                title="Daily Hospital Admission Rate from ED",
                labels={"ds":"Date","admission_rate":"Rate (0-1)"})
    fig.add_hline(y=admit_daily["admission_rate"].mean(), line_dash="dash",
                  annotation_text=f"Mean: {admit_daily['admission_rate'].mean():.2f}")
    fig.update_layout(template="plotly_white"); fig.show()
    print(f"Overall admission rate: {df['_admin_int'].mean():.1%}")
else:
    print("No patient_admin_flag column — skipping admission rate analysis.")

## Train / Validation / Test Split

In [ ]:
n=len(daily); test_start=n-TEST_DAYS; val_start=test_start-VAL_DAYS
ts_train=daily.iloc[:val_start].copy(); ts_val=daily.iloc[val_start:test_start].copy()
ts_test=daily.iloc[test_start:].copy(); ts_trainval=daily.iloc[:test_start].copy()
print(f"Train={len(ts_train)} | Val={len(ts_val)} | Test={len(ts_test)}")
assert ts_train["ds"].max()<ts_val["ds"].min()
assert ts_val["ds"].max()<ts_test["ds"].min()

## Feature Engineering

In [ ]:
def build_feats(df_in):
    df=df_in.copy().sort_values("ds").reset_index(drop=True); y=df["y"]
    for lag in [1,7,14,21]: df[f"lag_{lag}"]=y.shift(lag)
    for w in [7,14]: df[f"rmean_{w}"]=y.shift(1).rolling(w).mean(); df[f"rstd_{w}"]=y.shift(1).rolling(w).std()
    df["dow"]=df["ds"].dt.dayofweek; df["is_weekend"]=(df["dow"]>=5).astype(int)
    df["month"]=df["ds"].dt.month; df["quarter"]=df["ds"].dt.quarter
    return df
full_feat=build_feats(daily)
feat_train=full_feat.iloc[:val_start].dropna().copy()
feat_val=full_feat.iloc[val_start:test_start].dropna().copy()
feat_test=full_feat.iloc[test_start:].dropna().copy()
feat_trainval=full_feat.iloc[:test_start].dropna().copy()
FEAT_COLS=[c for c in feat_train.columns if c not in ["ds","y"]]
print(f"Features: {FEAT_COLS}")

## Baselines, LazyPredict, FLAML, StatsForecast, MLForecast

In [ ]:
results=[]; y_test=ts_test["y"].values; last_tv=ts_trainval["y"].values
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-1]),"Naive"))
sn=np.tile(last_tv[-SEASON_LEN:],TEST_DAYS//SEASON_LEN+1)[:TEST_DAYS]
results.append(metrics(y_test,sn,"Seasonal Naive (7-day)"))
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-7:].mean()),"MA-7"))

try:
    lz=LazyRegressor(verbose=0,ignore_warnings=True)
    lm,_=lz.fit(feat_train[FEAT_COLS],feat_val[FEAT_COLS],feat_train["y"],feat_val["y"])
    print("LazyPredict top 10:"); print(lm.sort_values("RMSE").head(10).to_string())
except Exception as e: print(f"LazyPredict: {e}")

flaml=AutoML()
flaml.fit(feat_trainval[FEAT_COLS],feat_trainval["y"],task="regression",metric="rmse",
          time_budget=FLAML_BUDGET,verbose=0,seed=RANDOM_STATE)
flaml_pred=flaml.predict(feat_test[FEAT_COLS]) if len(feat_test)>0 else None
if flaml_pred is not None:
    results.append(metrics(feat_test["y"].values,flaml_pred,f"FLAML ({flaml.best_estimator})"))

sf_df=ts_trainval[["ds","y"]].assign(unique_id="er")
sf=StatsForecast(models=[AutoARIMA(season_length=SEASON_LEN),AutoETS(season_length=SEASON_LEN),
                          AutoTheta(season_length=SEASON_LEN)],freq=FREQ,n_jobs=1)
sf.fit(sf_df); sf_fc=sf.forecast(h=HORIZON)
for col in ["AutoARIMA","AutoETS","AutoTheta"]:
    if col in sf_fc.columns:
        results.append(metrics(y_test,sf_fc[col].values[:TEST_DAYS],f"SF-{col}"))

from mlforecast import MLForecast
from lightgbm import LGBMRegressor
mlf_df=ts_trainval[["ds","y"]].assign(unique_id="er")
mlf=MLForecast(models={"lgbm":LGBMRegressor(n_estimators=200,learning_rate=0.05,
                                              verbosity=-1,random_state=RANDOM_STATE)},
               freq=FREQ,lags=[1,7,14],lag_transforms={1:[("rolling_mean",7)]},
               date_features=["dayofweek","month"],num_threads=2)
mlf.fit(mlf_df); mlf_fc=mlf.predict(h=HORIZON)
results.append(metrics(y_test,mlf_fc["lgbm"].values[:TEST_DAYS],"MLF-LightGBM"))

## Top 3 & Forecast

In [ ]:
res_df=pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string()); top3=res_df.head(3)
print("\n>>> TOP 3 <<<"); print(top3.to_string(index=False))
fig=px.bar(res_df,x="Model",y="RMSE",color="RMSE",color_continuous_scale="RdYlGn_r",
           title="ED Arrival Forecast — Model Comparison")
fig.update_layout(template="plotly_white",xaxis_tickangle=-35); fig.show()

In [ ]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=ts_trainval["ds"].tail(60),y=ts_trainval["y"].tail(60),
    name="Train",line=dict(color="royalblue",dash="dot",width=1)))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],
    name="Actual",line=dict(color="black",width=2)))
preds={}
if flaml_pred is not None: preds[f"FLAML ({flaml.best_estimator})"]=flaml_pred
for col in ["AutoARIMA","AutoETS","AutoTheta"]:
    if col in sf_fc.columns: preds[f"SF-{col}"]=sf_fc[col].values[:TEST_DAYS]
if "lgbm" in mlf_fc.columns: preds["MLF-LightGBM"]=mlf_fc["lgbm"].values[:TEST_DAYS]
preds["Seasonal Naive (7-day)"]=sn
colors=["#e15759","#f28e2b","#59a14f"]
for i,(_,row) in enumerate(top3.iterrows()):
    m=row["Model"]
    if m in preds:
        fig.add_trace(go.Scatter(x=ts_test["ds"][:len(preds[m])],y=preds[m],
            name=f"#{i+1} {m}",line=dict(color=colors[i],width=2)))
fig.update_layout(title="Top 3 ED Arrival Forecasts vs Actual",
                  template="plotly_white",xaxis_title="Date",yaxis_title="Patients")
fig.show()

## Error Analysis

In [ ]:
best_m=top3.iloc[0]["Model"]
if best_m in preds:
    bp=np.asarray(preds[best_m]).ravel(); ba=y_test[:len(bp)]; err=ba-bp
    print(f"Bias={err.mean():+.2f}  Std={err.std():.2f}")
    fig,ax=plt.subplots(1,2,figsize=(14,4))
    ax[0].hist(err,bins=15,color="steelblue",edgecolor="white")
    ax[0].axvline(0,color="red",linestyle="--"); ax[0].set_title("Error Distribution")
    ax[1].scatter(ba,bp,s=50,alpha=0.7,color="steelblue")
    lim=max(ba.max(),bp.max())*1.1; ax[1].plot([0,lim],[0,lim],"r--")
    ax[1].set_xlabel("Actual"); ax[1].set_ylabel("Predicted"); ax[1].set_title("Actual vs Predicted")
    plt.tight_layout(); plt.show()

## Insights & Interpretation
1. **Weekly seasonality is dominant** — Mondays often have highest ED volume (weekend illness presents Monday)
2. **Saturdays and Sundays** have different patterns: Saturday high (sports injuries, alcohol), Sunday lower
3. **Seasonal surge** in winter months (respiratory illness, slips/falls in cold weather)
4. **Wait time** correlates with arrival volume — a useful live metric for capacity alerting
5. **AutoARIMA/AutoTheta** typically outperform lag-feature ML on regular weekly series

## Limitations
1. No hour-of-day granularity — ED staffing is often based on 4-hour blocks, not just daily
2. No triage severity breakdown — 5 priority-1 patients needs more resource than 50 priority-5
3. No weather data — cold snap correlates with respiratory admissions
4. Single-site dataset — multi-ED systems need hierarchical reconciliation

## How to Improve
1. Use hourly rather than daily aggregation for shift-level staffing decisions
2. Add weather features (temperature, precipitation) from NOAA/OpenMeteo
3. Separate trauma (random, hard to predict) from illness visits (more seasonal)
4. Use quantile forecasting for high/low scenarios instead of point forecasts
5. Build a decision-support dashboard that converts forecast to staffing recommendation

## Common Mistakes
1. Using patient_id count vs visit count (a patient may have multiple visits)
2. Ignoring the distinction between ED arrivals and ED admissions (admitted ≠ arrived)
3. Using calendar day only — ED has pre-holiday and post-holiday patterns not captured by dow
4. MAPE fails on very low-volume nights (late Saturday → Sunday morning); use MAE instead

## Exercises
1. Resample to 4-hour blocks and build a within-day staffing forecast
2. Add a "bank holiday" binary flag and see impact on wait time forecast
3. Test whether wait_time is a useful leading indicator (lagged by 1 day) for next-day volume
4. Compute rolling-origin MAE over 10 test windows for more robust evaluation

## Summary
- Aggregated individual ED visit records to daily arrival counts
- EDA: strong weekly and seasonal patterns; wait time analysis
- Compared Baselines → LazyPredict → FLAML → StatsForecast → MLForecast
- Key lesson: ED forecasting is a life-critical application — uncertainty bands matter
  as much as point accuracy; always use a conservative staffing buffer

---
*Notebook #14 of 50 — Time Series Forecasting Portfolio*